In [258]:
from datetime import datetime, timedelta
import backtrader as bt
import pandas as pd

class MyStrategy(bt.Strategy):
    def __init__(self):
        # przypisanie feedów do zmiennych w strategii
        self.data_m1  = self.datas[0]  # pierwszy dodany feed
        self.data_m15 = self.datas[1]  # drugi feed
        self.data_h1 = self.datas[2]  # drugi feed
        self.data_h4  = self.datas[3]  # trzeci feed

        self.last_execution_order = datetime.now() - timedelta(days=2000)
        self.STOP_LOSS_DIFF = 0.1
        self.TAKE_PROFIT_DIFF = 0.1
        self.DEFAULT_POSITION_SIZE = 50
        self.start_price = None
        self.START_HOUR = 8
        self.END_HOUR = 17
        self.POSITION_BRAKE_DELTA = timedelta(minutes=15)

    def next(self):
        last10_h4 = list(self.data_h4.close.get(size=10))
        if self.can_open_new_position() and len(last10_h4) > 2:
            last10_m1 = list(self.data_m1.close.get(size=10))
            last10_m15 = list(self.data_m15.close.get(size=10))
            last10_h4 = list(self.data_h4.close.get(size=10))
            if last10_h4[0] > last10_h4[1] > last10_h4[2]:
                if last10_m1[0] > last10_m1[1] > last10_m1[2]:
                    self.buy(size=self.DEFAULT_POSITION_SIZE)
            elif last10_h4[0] < last10_h4[1] < last10_h4[2]:
                if last10_m1[0] < last10_m1[1] < last10_m1[2]:
                    self.sell(size=self.DEFAULT_POSITION_SIZE)

        # if self.can_open_new_position():
        #     print("OPEN NEW POSITION")

            # print("XXXXXZZZZ")
            # print(self.position.price)
            # self.sell(exectype=bt.Order.Stop, price=self.data_m1.close[0] - 0.1)
            # self.buy_bracket(size=50, stopprice=self.data_m1.close[0] - 10)
            # print(self.data_m1.close[0])

        # target_close_date = datetime(2025, 10, 24, 12, 9, 0)
        # if self.position and current_date == target_close_date:
        #     print(self.position.size)
        #     print(self.position.price)
        #     print(self.broker.orders)
        #     print([x.getstatusname() for x in self.broker.orders])
        #     print([x.price for x in self.broker.orders])
        #     # self.close()

    def notify_order(self, order):
        if order.status == order.Completed:
            size = self.position.size
            executed_size = order.executed.size
            executed_price = order.executed.price
            if size == 0:
                print("Order Completed = position = 0")
                for o in self.broker.get_orders_open():
                    self.cancel(o)
                self.last_execution_order = self._get_current_date()

            if size == executed_size:
                self.start_price = executed_price
                print(self._get_current_date())
                if size > 0:
                    print(f"Order Completed = position > 0 = {size}")
                    self.sell(exectype=bt.Order.Stop, price=executed_price - self.STOP_LOSS_DIFF, size=executed_size)
                    self.sell(exectype=bt.Order.Limit, price=executed_price + self.TAKE_PROFIT_DIFF, size=executed_size)
                elif size < 0:
                    print(f"Order Completed = position < 0 = {size}")
                    self.buy(exectype=bt.Order.Stop, price=executed_price + self.STOP_LOSS_DIFF, size=executed_size)
                    self.buy(exectype=bt.Order.Limit, price=executed_price - self.TAKE_PROFIT_DIFF, size=executed_size)

    def can_open_new_position(self):
        current_date = self._get_current_date()
        is_hour_right = current_date.hour >= self.START_HOUR and current_date.hour <= self.END_HOUR
        was_last_position_a_while_ago = current_date - self.POSITION_BRAKE_DELTA > self.last_execution_order
        return self.position.size == 0 and is_hour_right and was_last_position_a_while_ago

    def _get_current_date(self):
        return bt.num2date(self.data_m1.datetime[-1])

In [259]:
def load_csv(path):
    df = pd.read_csv(path)
    df["open"]  = df["low"] + df["delta_open"]
    df["close"] = df["low"] + df["delta_close"]
    df["high"]  = df["low"] + df["delta_high"]
    df['datetime'] = pd.to_datetime(df['timestamp'])
    df = df.set_index('datetime').sort_index()
    df = df[["open", "high", "low", "close", "volume"]]
    return bt.feeds.PandasData(dataname=df)

def test_strategy(strategy):

    data_m1  = load_csv("data/XTIUSD_20251018_2200_20251025_2159_M1.csv")
    data_m15 = load_csv("data/XTIUSD_20251018_2200_20251025_2159_M15.csv")
    data_h1 = load_csv("data/XTIUSD_20251018_2200_20251025_2159_H1.csv")
    data_h4  = load_csv("data/XTIUSD_20251018_2200_20251025_2159_H4.csv")

    cerebro = bt.Cerebro()
    cerebro.adddata(data_m1)
    cerebro.adddata(data_m15)
    cerebro.adddata(data_h1)
    cerebro.adddata(data_h4)

    cerebro.addstrategy(strategy)
    cerebro.broker.setcash(100000)

    return cerebro ,cerebro.run()

In [260]:
cerebro, results = test_strategy(MyStrategy)
print("Starting Portfolio Value:", cerebro.broker.getvalue())
print("Final Portfolio Value:", cerebro.broker.getvalue())

# === 5. Dostęp do analiz ===
strat = results[0]
analyzers = strat.analyzers

if hasattr(analyzers, 'trades'):
    print("Number of trades:", analyzers.trades.get_analysis().get('total', 0))

if hasattr(analyzers, 'sharpe'):
    print("Sharpe Ratio:", analyzers.sharpe.get_analysis())


2025-10-21 09:02:00
Order Completed = position > 0 = 50
Order Completed = position = 0
2025-10-21 09:59:00
Order Completed = position > 0 = 50
Order Completed = position = 0
2025-10-21 10:32:00
Order Completed = position > 0 = 50
Order Completed = position = 0
2025-10-21 11:32:00
Order Completed = position > 0 = 50
Order Completed = position = 0
2025-10-21 12:05:00
Order Completed = position > 0 = 50
Order Completed = position = 0
2025-10-21 12:34:00
Order Completed = position > 0 = 50
Order Completed = position = 0
2025-10-21 13:00:00
Order Completed = position > 0 = 50
Order Completed = position = 0
2025-10-21 13:27:00
Order Completed = position > 0 = 50
Order Completed = position = 0
2025-10-21 13:46:00
Order Completed = position > 0 = 50
Order Completed = position = 0
2025-10-21 14:07:00
Order Completed = position > 0 = 50
Order Completed = position = 0
2025-10-21 14:31:00
Order Completed = position > 0 = 50
Order Completed = position = 0
2025-10-21 15:16:00
Order Completed = posit